In [31]:
import pandas as pd

csv_path = "/kaggle/input/datasets/saumyajitdas1/csv-file/base_data.csv"

df = pd.read_csv(csv_path)

print(df.head())
print(df.shape)
print(df.columns)

   face_happy  face_sad  face_angry  face_fear  face_surprise  face_neutral  \
0       0.866     0.031       0.012      0.601          0.708         0.750   
1       0.525     0.037       0.061      0.432          0.291         0.608   
2       0.514     0.157       0.040      0.592          0.046         0.845   
3       0.098     0.162       0.061      0.684          0.440         0.843   
4       0.520     0.133       0.062      0.547          0.185         0.649   

   scroll_speed  time_spent  interaction_count  quiz_response_time    label  
0         0.485       0.893              0.799               0.247  Focused  
1         0.491       0.933              0.606               0.255  Focused  
2         0.242       0.717              0.683               0.337  Focused  
3         0.251       0.626              0.974               0.490  Focused  
4         0.349       0.614              0.955               0.278  Focused  
(500, 11)
Index(['face_happy', 'face_sad', 'face_angry', 

In [32]:
label_map = {
    "Focused": 0,
    "Confused": 1,
    "Bored": 2,
    "Frustrated": 3,
    "Distracted": 4
}

df["label"] = df["label"].map(label_map)

print("Mapped labels:", df["label"].unique())

Mapped labels: [0 1 2 3 4]


In [33]:
print(df.isnull().sum())

face_happy            0
face_sad              0
face_angry            0
face_fear             0
face_surprise         0
face_neutral          0
scroll_speed          0
time_spent            0
interaction_count     0
quiz_response_time    0
label                 0
dtype: int64


In [34]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

features = df.iloc[:, :-1].values
labels = df.iloc[:, -1].values

features = scaler.fit_transform(features)

In [35]:
import torch
from torch.utils.data import Dataset

class EngagementDataset(Dataset):
    def __init__(self, features, labels):
        self.X = torch.tensor(features, dtype=torch.float32)
        self.y = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [36]:
# Remove rows where label is NaN
df = df[df.iloc[:, -1].notna()]

# Convert label column to integer safely
df.iloc[:, -1] = pd.to_numeric(df.iloc[:, -1], errors='coerce')

# Drop any rows that still failed conversion
df = df.dropna()

# Convert to int
df.iloc[:, -1] = df.iloc[:, -1].astype(int)

df = df.reset_index(drop=True)

print("Fixed Labels:", df.iloc[:, -1].unique())

Fixed Labels: [0 1 2 3 4]


In [37]:
dataset = EngagementDataset(features, labels)

In [38]:
import torch

features = df.iloc[:, :-1].values
labels = df.iloc[:, -1].values

print("Feature shape:", features.shape)
print("Label shape:", labels.shape)
print("Label sample:", labels[:10])

Feature shape: (500, 10)
Label shape: (500,)
Label sample: [0 0 0 0 0 0 0 0 0 0]


In [39]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset

train_idx, val_idx = train_test_split(
    list(range(len(dataset))),
    test_size=0.2,
    random_state=42
)

train_dataset = Subset(dataset, train_idx)
val_dataset = Subset(dataset, val_idx)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [40]:
input_dim = dataset[0][0].shape[0]
print("Input Dimension:", input_dim)

Input Dimension: 10


In [41]:
model = EngagementModel(input_dim=input_dim).to(device)

In [42]:
print("Dataset size:", len(dataset))
print("Sample X shape:", dataset[0][0].shape)
print("Sample label:", dataset[0][1])

Dataset size: 500
Sample X shape: torch.Size([10])
Sample label: tensor(0)


In [43]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset

train_idx, val_idx = train_test_split(
    list(range(len(dataset))),
    test_size=0.2,
    random_state=42
)

train_dataset = Subset(dataset, train_idx)
val_dataset = Subset(dataset, val_idx)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))

Train size: 400
Val size: 100


In [44]:
import torch.nn as nn

class EngagementModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(32, 5)
        )

    def forward(self, x):
        return self.model(x)

In [45]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

input_dim = dataset[0][0].shape[0]
model = EngagementModel(input_dim).to(device)

print("Using device:", device)

Using device: cpu


In [46]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [47]:
def train_model(model, train_loader, val_loader, epochs=20):
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        correct = 0
        total = 0

        for X, y in train_loader:
            X, y = X.to(device), y.to(device)

            optimizer.zero_grad()
            outputs = model(X)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == y).sum().item()
            total += y.size(0)

        train_acc = correct / total

        # Validation
        model.eval()
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(device), y.to(device)
                outputs = model(X)
                _, predicted = torch.max(outputs, 1)
                val_correct += (predicted == y).sum().item()
                val_total += y.size(0)

        val_acc = val_correct / val_total

        print(f"Epoch [{epoch+1}/{epochs}] "
              f"Loss: {total_loss:.4f} "
              f"Train Acc: {train_acc:.4f} "
              f"Val Acc: {val_acc:.4f}")

In [48]:
train_model(model, train_loader, val_loader)

Epoch [1/20] Loss: 20.6704 Train Acc: 0.2725 Val Acc: 0.4300
Epoch [2/20] Loss: 19.0256 Train Acc: 0.5200 Val Acc: 0.7500
Epoch [3/20] Loss: 16.8602 Train Acc: 0.7500 Val Acc: 0.9500
Epoch [4/20] Loss: 14.4406 Train Acc: 0.8375 Val Acc: 0.9800
Epoch [5/20] Loss: 11.8737 Train Acc: 0.8750 Val Acc: 0.9700
Epoch [6/20] Loss: 8.9982 Train Acc: 0.8950 Val Acc: 1.0000
Epoch [7/20] Loss: 7.0312 Train Acc: 0.9075 Val Acc: 1.0000
Epoch [8/20] Loss: 5.2009 Train Acc: 0.9625 Val Acc: 1.0000
Epoch [9/20] Loss: 3.8147 Train Acc: 0.9575 Val Acc: 1.0000
Epoch [10/20] Loss: 3.0843 Train Acc: 0.9725 Val Acc: 1.0000
Epoch [11/20] Loss: 2.5861 Train Acc: 0.9675 Val Acc: 1.0000
Epoch [12/20] Loss: 1.9131 Train Acc: 0.9725 Val Acc: 1.0000
Epoch [13/20] Loss: 1.5538 Train Acc: 0.9725 Val Acc: 1.0000
Epoch [14/20] Loss: 1.3860 Train Acc: 0.9850 Val Acc: 1.0000
Epoch [15/20] Loss: 1.1844 Train Acc: 0.9850 Val Acc: 1.0000
Epoch [16/20] Loss: 0.9719 Train Acc: 0.9900 Val Acc: 1.0000
Epoch [17/20] Loss: 0.8899 T

In [49]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np

def evaluate_model(model, loader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            outputs = model(X)
            _, predicted = torch.max(outputs, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    return acc, all_labels, all_preds


# Evaluate on train
train_acc, train_labels, train_preds = evaluate_model(model, train_loader)

# Evaluate on validation
val_acc, val_labels, val_preds = evaluate_model(model, val_loader)

print("🔥 FINAL RESULTS")
print("Train Accuracy:", round(train_acc * 100, 2), "%")
print("Validation Accuracy:", round(val_acc * 100, 2), "%")

🔥 FINAL RESULTS
Train Accuracy: 100.0 %
Validation Accuracy: 100.0 %


In [50]:
label_decoder = {
    0: "Focused",
    1: "Confused",
    2: "Bored",
    3: "Frustrated",
    4: "Distracted"
}

In [51]:
import numpy as np
import torch
import torch.nn.functional as F

# Generate random sample (10 features)
random_sample = np.random.rand(1, 10)

# Scale it using SAME scaler
random_sample_scaled = scaler.transform(random_sample)

# Convert to tensor
input_tensor = torch.tensor(random_sample_scaled, dtype=torch.float32).to(device)

# Inference
model.eval()
with torch.no_grad():
    output = model(input_tensor)
    probabilities = F.softmax(output, dim=1)
    predicted_class = torch.argmax(probabilities, dim=1).item()

print("Random Input:", random_sample)
print("Predicted Class Index:", predicted_class)
print("Predicted Engagement State:", label_decoder[predicted_class])
print("Confidence:", round(probabilities[0][predicted_class].item() * 100, 2), "%")

Random Input: [[0.64780552 0.29438866 0.02043558 0.96186127 0.60178986 0.73837754
  0.90903787 0.54729142 0.49644973 0.6971615 ]]
Predicted Class Index: 2
Predicted Engagement State: Bored
Confidence: 81.72 %


In [52]:
torch.save(model.state_dict(), "engagement_model.pth")
print("Model saved successfully!")

Model saved successfully!


In [53]:
import joblib

joblib.dump(scaler, "scaler.pkl")
print("Scaler saved successfully!")

Scaler saved successfully!
